In [1]:
import pandas as pd

shipments = pd.read_csv("../02-pandas/shipments.csv")

shipments.head()

,Shipment_ID,City,Customer,Revenue,Delivery_Days,Status
0,1001,Berlin,Amazon,1200,2,Delivered
1,1002,Hamburg,Zalando,850,4,Delivered
2,1003,Munich,Amazon,1500,3,Delayed
3,1004,Berlin,Otto,650,1,Delivered
4,1005,Hamburg,Amazon,2100,5,Delayed


In [2]:
shipments.groupby("Customer")["Revenue"].sum()

Customer
Amazon     4800
Otto       1750
Zalando    3550
Name: Revenue, dtype: int64

### Exercise 02.04.02 — groupby() Fundamentals

In [3]:
shipments.groupby("City")["Revenue"].sum()

City
Berlin     2950
Hamburg    4700
Munich     2450
Name: Revenue, dtype: int64

In [4]:
shipments.groupby("City")["Delivery_Days"].mean()

City
Berlin     2.0
Hamburg    5.0
Munich     2.5
Name: Delivery_Days, dtype: float64

### Exercise 02.04.03 — Aggregation Functions
Task A — Shipment Volume

Calculate the number of shipments for each customer using .groupby() and .size().

Task B — Maximum Delivery Time

Calculate the longest delivery time recorded for each city.

Task C — Unique Customers

Calculate how many distinct customers have shipments in each city.

In [5]:
shipments.groupby("Customer").size()

Customer
Amazon     3
Otto       2
Zalando    3
dtype: int64

In [6]:
shipments.groupby("City")["Delivery_Days"].max()

City
Berlin     3
Hamburg    6
Munich     3
Name: Delivery_Days, dtype: int64

In [7]:
shipments.groupby("City")["Customer"].nunique()

City
Berlin     2
Hamburg    2
Munich     2
Name: Customer, dtype: int64

### Exercise 02.04.04 — Multiple Aggregations with .agg()

In [8]:
shipments.groupby("Customer").agg({
    "Revenue":["mean", "sum"],
    "Shipment_ID":["count"],
    "Delivery_Days": ["mean","max"]
})

Revenue       Shipment_ID Delivery_Days    
                 mean   sum       count          mean max
Customer                                                 
Amazon    1600.000000  4800           3      3.333333   5
Otto       875.000000  1750           2      2.000000   3
Zalando   1183.333333  3550           3      4.000000   6

### Exercise 02.04.05 — Grouping by Multiple Columns

In [9]:
shipments.groupby(["Customer", "City"])["Revenue"].sum().reset_index()

,Customer,City,Revenue
0,Amazon,Berlin,1200
1,Amazon,Hamburg,2100
2,Amazon,Munich,1500
3,Otto,Berlin,1750
4,Zalando,Hamburg,2600
5,Zalando,Munich,950


In [10]:
delivery_performance = shipments.groupby(["Customer","City"])["Delivery_Days"].mean().reset_index()

In [11]:
delivery_performance

,Customer,City,Delivery_Days
0,Amazon,Berlin,2.0
1,Amazon,Hamburg,5.0
2,Amazon,Munich,3.0
3,Otto,Berlin,2.0
4,Zalando,Hamburg,5.0
5,Zalando,Munich,2.0


### Exercise 02.04.06 — Named Aggregations

In [12]:
customer_kpis = shipments.groupby("Customer").agg(
    Total_Revenue = ("Revenue","sum"),
    Avg_Revenue = ("Revenue", "mean"),
    Shipments = ("Shipment_ID", "count"),
    Avg_Delivery_Days = ("Delivery_Days","mean"),
    Max_Delivery_Days = ("Delivery_Days","max")
)

In [13]:
customer_kpis

,Total_Revenue,Avg_Revenue,Shipments,Avg_Delivery_Days,Max_Delivery_Days
Customer,,,,,
Amazon,4800,1600.000000,3,3.333333,5
Otto,1750,875.000000,2,2.000000,3
Zalando,3550,1183.333333,3,4.000000,6


In [14]:
customer_kpis = customer_kpis.reset_index()

In [42]:
customer_kpis

,Customer,Total_Revenue,Avg_Revenue,Shipments,Avg_Delivery_Days,Max_Delivery_Days
0,Amazon,4800,1600.000000,3,3.333333,5
1,Otto,1750,875.000000,2,2.000000,3
2,Zalando,3550,1183.333333,3,4.000000,6


### Exercise 02.04.07 — Business KPI Calculations

In [16]:
shipments

,Shipment_ID,City,Customer,Revenue,Delivery_Days,Status
0,1001,Berlin,Amazon,1200,2,Delivered
1,1002,Hamburg,Zalando,850,4,Delivered
2,1003,Munich,Amazon,1500,3,Delayed
3,1004,Berlin,Otto,650,1,Delivered
4,1005,Hamburg,Amazon,2100,5,Delayed
5,1006,Munich,Zalando,950,2,Delivered
6,1007,Berlin,Otto,1100,3,Delivered
7,1008,Hamburg,Zalando,1750,6,Delayed


In [17]:
delayed_shipments = (shipments["Status"] == "Delayed").sum()
total_shipments = len(shipments)

In [18]:
total_shipments

8

In [19]:
delayed_rate = (delayed_shipments / total_shipments * 100)
delayed_rate

np.float64(37.5)

In [20]:
delayed_rate = (shipments["Status"] == "Delayed").mean() * 100
delayed_rate

np.float64(37.5)

### Exercise 02.04.07 — Task B: Delayed Shipment Rate by Customer

In [21]:
shipments["Is_Delayed"] = (shipments["Status"] == "Delayed")

In [22]:
customer_delay_rates = shipments.groupby("Customer")["Is_Delayed"].mean()*100

In [23]:
customer_delay_rates

Customer
Amazon     66.666667
Otto        0.000000
Zalando    33.333333
Name: Is_Delayed, dtype: float64

### Exercise 02.04.07 — Task C: Preparing the KPI Report

In [24]:
customer_delay_rates = customer_delay_rates.reset_index()
customer_delay_rates = customer_delay_rates.rename(columns={"Is_Delayed":"Delayed_Rate_Pct"})
customer_delay_rates["Delayed_Rate_Pct"] = (
    customer_delay_rates["Delayed_Rate_Pct"].round(2)
)



In [25]:
customer_delay_rates

,Customer,Delayed_Rate_Pct
0,Amazon,66.67
1,Otto,0.00
2,Zalando,33.33


### Exercise 02.04.07 — Task D: Revenue Contribution (%)

In [26]:
total_company_revenue = shipments.groupby("Customer")["Revenue"].sum()

In [37]:
shipments.groupby("Customer")["Revenue"].sum()

Customer
Amazon     4800
Otto       1750
Zalando    3550
Name: Revenue, dtype: int64

In [49]:
total_company_revenue = customer_kpis["Total_Revenue"].sum()

customer_kpis["Revenue_Share_Pct"] = (customer_kpis["Total_Revenue"] / total_company_revenue *100).round(2)

In [50]:
customer_kpis

,Customer,Total_Revenue,Avg_Revenue,Shipments,Avg_Delivery_Days,Max_Delivery_Days,Revenue_Share_Pct
0,Amazon,4800,1600.000000,3,3.333333,5,47.52
1,Otto,1750,875.000000,2,2.000000,3,17.33
2,Zalando,3550,1183.333333,3,4.000000,6,35.15


### Exercise 02.04.08 — Sorting & Ranking

In [51]:
customer_kpis.sort_values("Total_Revenue")

,Customer,Total_Revenue,Avg_Revenue,Shipments,Avg_Delivery_Days,Max_Delivery_Days,Revenue_Share_Pct
1,Otto,1750,875.000000,2,2.000000,3,17.33
2,Zalando,3550,1183.333333,3,4.000000,6,35.15
0,Amazon,4800,1600.000000,3,3.333333,5,47.52


In [54]:
top_customers = customer_kpis.sort_values("Total_Revenue", ascending = False).head(2)

In [55]:
top_customers

,Customer,Total_Revenue,Avg_Revenue,Shipments,Avg_Delivery_Days,Max_Delivery_Days,Revenue_Share_Pct
0,Amazon,4800,1600.000000,3,3.333333,5,47.52
2,Zalando,3550,1183.333333,3,4.000000,6,35.15


### Exercise 02.04.08 — Task B: Ranking Delivery Performance

In [59]:
slowest_customers = customer_kpis.sort_values("Avg_Delivery_Days", ascending=False).head(2)

In [60]:
slowest_customers

,Customer,Total_Revenue,Avg_Revenue,Shipments,Avg_Delivery_Days,Max_Delivery_Days,Revenue_Share_Pct
2,Zalando,3550,1183.333333,3,4.000000,6,35.15
0,Amazon,4800,1600.000000,3,3.333333,5,47.52


### Exercise 02.04.09 — Pivot Tables

In [64]:
revenue_pivot = pd.pivot_table(
    shipments,
    index = "Customer",
    columns = "City",
    values = "Revenue",
    aggfunc = "sum",
    fill_value = 0
)

In [65]:
revenue_pivot

City,Berlin,Hamburg,Munich
Customer,,,
Amazon,1200,2100,1500
Otto,1750,0,0
Zalando,0,2600,950


### Exercise 02.04.09 — Task B: Pivot Tables with Totals

In [67]:
revenue_pivot_totals = pd.pivot_table(
    shipments,
    index = "Customer",
    columns = "City",
    values = "Revenue",
    aggfunc = "sum",
    fill_value = 0,
    margins = True
)

In [68]:
revenue_pivot_totals

City,Berlin,Hamburg,Munich,All
Customer,,,,
Amazon,1200,2100,1500,4800
Otto,1750,0,0,1750
Zalando,0,2600,950,3550
All,2950,4700,2450,10100


### Exercise 02.04.09 — Task C: Delivery Performance Pivot Table

In [70]:
shipments.groupby(["Customer", "City"])["Delivery_Days"].mean().reset_index()

,Customer,City,Delivery_Days
0,Amazon,Berlin,2.0
1,Amazon,Hamburg,5.0
2,Amazon,Munich,3.0
3,Otto,Berlin,2.0
4,Zalando,Hamburg,5.0
5,Zalando,Munich,2.0


In [75]:
delivery_pivot = pd.pivot_table(
    shipments,
    index = "Customer",
    columns = "City",
    values = "Delivery_Days",
    aggfunc = "mean",
)

In [76]:
delivery_pivot

City,Berlin,Hamburg,Munich
Customer,,,
Amazon,2.0,5.0,3.0
Otto,2.0,NaN,NaN
Zalando,NaN,5.0,2.0


### Exercise 02.04.10 — Final Business Analytics Challenge

In [81]:
customer_scorecard = shipments.groupby(["Customer"]).agg(
    Total_Revenue = ("Revenue", "sum"),
    Total_Shipments = ("Shipment_ID", "count"),
    Avg_Revenue = ("Revenue", "mean"),
    Avg_Delivery_Days = ("Delivery_Days","mean"),    
).reset_index()

In [82]:
customer_scorecard

,Customer,Total_Revenue,Total_Shipments,Avg_Revenue,Avg_Delivery_Days
0,Amazon,4800,3,1600.000000,3.333333
1,Otto,1750,2,875.000000,2.000000
2,Zalando,3550,3,1183.333333,4.000000


In [97]:
shipments["Is_Delayed"] = shipments["Status"] == "Delayed"
shipments

,Shipment_ID,City,Customer,Revenue,Delivery_Days,Status,Is_Delayed,Revenue_Share_Pct
0,1001,Berlin,Amazon,1200,2,Delivered,False,NaN
1,1002,Hamburg,Zalando,850,4,Delivered,False,NaN
2,1003,Munich,Amazon,1500,3,Delayed,True,NaN
3,1004,Berlin,Otto,650,1,Delivered,False,NaN
4,1005,Hamburg,Amazon,2100,5,Delayed,True,NaN
5,1006,Munich,Zalando,950,2,Delivered,False,NaN
6,1007,Berlin,Otto,1100,3,Delivered,False,NaN
7,1008,Hamburg,Zalando,1750,6,Delayed,True,NaN


In [ ]:
shipments.groupby("Customer")["Status"].count()

Customer
Amazon     3
Otto       2
Zalando    3
Name: Status, dtype: int64

In [98]:
delayed_counts = shipments.groupby("Customer")["Is_Delayed"].sum()
delayed_counts

Customer
Amazon     2
Otto       0
Zalando    1
Name: Is_Delayed, dtype: int64

In [99]:
customer_scorecard

,Customer,Total_Revenue,Total_Shipments,Avg_Revenue,Avg_Delivery_Days
0,Amazon,4800,3,1600.000000,3.333333
1,Otto,1750,2,875.000000,2.000000
2,Zalando,3550,3,1183.333333,4.000000


In [100]:
customer_scorecard["Delayed_Shipments"] = customer_scorecard["Customer"].map(delayed_counts)

In [101]:
customer_scorecard

,Customer,Total_Revenue,Total_Shipments,Avg_Revenue,Avg_Delivery_Days,Delayed_Shipments
0,Amazon,4800,3,1600.000000,3.333333,2
1,Otto,1750,2,875.000000,2.000000,0
2,Zalando,3550,3,1183.333333,4.000000,1


In [106]:
customer_scorecard["Delay_Rate_Pct"] = (customer_scorecard["Delayed_Shipments"] / customer_scorecard["Total_Shipments"]*100).round(2)

In [107]:
customer_scorecard

,Customer,Total_Revenue,Total_Shipments,Avg_Revenue,Avg_Delivery_Days,Delayed_Shipments,Delay_Rate_Pct
0,Amazon,4800,3,1600.000000,3.333333,2,66.67
1,Otto,1750,2,875.000000,2.000000,0,0.00
2,Zalando,3550,3,1183.333333,4.000000,1,33.33


In [118]:
total_company_revenue = customer_scorecard["Total_Revenue"].sum()
customer_scorecard["Revenue_Share_Pct"] = (customer_scorecard["Total_Revenue"] / total_company_revenue * 100).round(2)

In [119]:
customer_scorecard

,Customer,Total_Revenue,Total_Shipments,Avg_Revenue,Avg_Delivery_Days,Delayed_Shipments,Delay_Rate_Pct,Revenue_Share_Pct
0,Amazon,4800,3,1600.000000,3.333333,2,66.67,47.52
1,Otto,1750,2,875.000000,2.000000,0,0.00,17.33
2,Zalando,3550,3,1183.333333,4.000000,1,33.33,35.15


In [122]:
final_scorecard = customer_scorecard.sort_values("Total_Revenue", ascending=False).reset_index(drop=True)

In [123]:
final_scorecard

,Customer,Total_Revenue,Total_Shipments,Avg_Revenue,Avg_Delivery_Days,Delayed_Shipments,Delay_Rate_Pct,Revenue_Share_Pct
0,Amazon,4800,3,1600.000000,3.333333,2,66.67,47.52
1,Zalando,3550,3,1183.333333,4.000000,1,33.33,35.15
2,Otto,1750,2,875.000000,2.000000,0,0.00,17.33
